In [ ]:
import os
import json
import sqlite3
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
Model = "gpt-4.1-mini"

System_Message = """ 
                    You are a helpful assistant who keeps track of user progress
                    Suggest user what can be done. Motivate user to achieve more.
                    You should provide in detail analysis what is achieved and how can a 
                    user do better. If you dont have any answer for the question given ask for
                    more clarification and think. Apart from if there are any general questions related
                    to software engineering please be mindful and help.
                    
                    If user asks for summary of the progress or analysis of the progress
                    Provide summary for every date what topics were covered time spent and if more time could be spent
                    And provide entire week summary as well. And compare all the weeks progress and let user know if there
                    are any lapses"""
Messages = [
    {
        "role":"system",
        "content":System_Message
    }
]


In [ ]:
#Making Db Connection
DB = "progress.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    # "AUTOINCREMENT" ensures IDs are never reused even if rows are deleted
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS progress(
            id INTEGER PRIMARY KEY AUTOINCREMENT, 
            Date TEXT NOT NULL, 
            Topics_Learnt TEXT, 
            Topic_Category TEXT, 
            Time_Spent_Minutes INTEGER
        )
    ''')
    conn.commit()

In [ ]:
def getProgreessByDate(date):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM progress WHERE Date = ?',(date,))
        result = cursor.fetchone()
        return result

def saveProgress(Date,TopicLearnt,TopicCategory,TimeSpent):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        query = '''
            INSERT INTO learning_log (Date, Topics_Learnt, Topic_Category, Time_Spent_Minutes)
            VALUES (?, ?, ?, ?)'''
        cursor.execute(query,(Date,TopicLearnt,TopicCategory,TimeSpent))
        cursor.commit()

In [ ]:
def getAllProgress():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM progress")
        result = cursor.fetchall()
        return result

In [ ]:
saveProgress = {
    "name": "saveProgress",
    "description": "Save the user's learning progress into the database.",
    "parameters": {
        "type": "object",
        "properties": {
            "Date": {
                "type": "string",
                "description": "The date of the entry in YYYY-MM-DD format.",
            },
            "TopicLearnt": {
                "type": "string",
                "description": "A detailed description of the specific topic studied.",
            },
            "TopicCategory": {
                "type": "string",
                "description": "The broad category of the topic (e.g., Programming, Math, Design).",
            },
            "TimeSpent": {
                "type": "integer",
                "description": "The total duration of the study session in minutes.",
            }
        },
        "required": ["Date", "TopicLearnt", "TopicCategory", "TimeSpent"],
        "additionalProperties": False
    }
}

import openai


def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": System_Message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=Model, messages=messages, tools=tools)  

    print(response.choices[0])
    
    return "Aadil Shaik"

In [ ]:
gr.ChatInterface(
    fn = chat,
    type = "messages"
).launch()